In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    silhouette_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    r2_score,
)

# For reproducibility
RANDOM_STATE = 42

In [3]:
# Update path to your CSV file
csv_path = "pantherella_sales_uk.csv"
df = pd.read_csv(csv_path)

# Quick peek
df.head()

,order_id,order_date,region,city,product_name,product_category,unit_price,quantity,total_sales
0,1001,2024-01-05,England,London,Danvers Merino Wool Sock,Merino Wool,25.0,3,75.0
1,1002,2024-01-07,Scotland,Edinburgh,Laburnum Egyptian Cotton Sock,Cotton,22.0,2,44.0
2,1003,2024-01-10,England,Manchester,Danvers Merino Wool Sock,Merino Wool,25.0,1,25.0
3,1004,2024-01-12,Wales,Cardiff,Silk Blend Dress Sock,Silk Blend,28.0,2,56.0
4,1005,2024-01-14,England,Birmingham,Laburnum Egyptian Cotton Sock,Cotton,22.0,4,88.0


In [4]:
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())

Columns: ['order_id', 'order_date', 'region', 'city', 'product_name', 'product_category', 'unit_price', 'quantity', 'total_sales']

Data types:
 order_id              int64
order_date           object
region               object
city                 object
product_name         object
product_category     object
unit_price          float64
quantity              int64
total_sales         float64
dtype: object

Missing values per column:
 order_id            0
order_date          0
region              0
city                0
product_name        0
product_category    0
unit_price          0
quantity            0
total_sales         0
dtype: int64


In [5]:
df = df.rename(columns={
    'Sales': 'sales_value',
    'ProfitMargin': 'profit_margin',
    # ...
})

In [8]:
df.columns

Index(['order_id', 'order_date', 'region', 'city', 'product_name',
       'product_category', 'unit_price', 'quantity', 'total_sales'],
      dtype='object')

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Clean column names (removes spaces and standardizes formatting)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Check available columns
print("Columns in dataset:", df.columns)

# Verify the column exists before plotting
if 'profit_margin' in df.columns:
    
    plt.figure(figsize=(8,4))
    
    sns.histplot(
        data=df,
        x='profit_margin',
        bins=50,
        kde=True
    )
    
    plt.title('Distribution of Profit Margin')
    plt.xlabel('Profit Margin (%) or Unit')
    plt.ylabel('Count')
    
    plt.show()

else:
    print("Column 'profit_margin' not found. Available columns:", df.columns)

Columns in dataset: Index(['order_id', 'order_date', 'region', 'city', 'product_name',
       'product_category', 'unit_price', 'quantity', 'total_sales'],
      dtype='object')
Column 'profit_margin' not found. Available columns: Index(['order_id', 'order_date', 'region', 'city', 'product_name',
       'product_category', 'unit_price', 'quantity', 'total_sales'],
      dtype='object')


In [12]:
features_for_clustering = []

if 'sales_value' in df.columns:
    features_for_clustering.append('sales_value')

if 'profit_margin' in df.columns:
    features_for_clustering.append('profit_margin')

if 'age' in df.columns:
    features_for_clustering.append('age')

categorical_cols = []

for col in ['gender', 'region', 'income_band']:
    if col in df.columns:
        categorical_cols.append(col)

df_encoded = pd.get_dummies(df[features_for_clustering + categorical_cols], drop_first=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

In [13]:
sil_scores = []
k_options = range(2, 8)  # try 2–7 clusters
for k in k_options:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    
# Display
for k, score in zip(k_options, sil_scores):
    print(f"k={k}, silhouette score={score:.3f}")

k=2, silhouette score=0.628
k=3, silhouette score=0.803
k=4, silhouette score=1.000
k=5, silhouette score=1.000
k=6, silhouette score=1.000
k=7, silhouette score=1.000


/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Users/adnanaltimeemy/miniconda3/envs/coding/lib/python3.12/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (7). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [14]:
best_k = 3  # replace with chosen value
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Count per cluster
df['cluster'].value_counts().sort_index()

cluster
0    14
1     2
2     4
Name: count, dtype: int64

In [19]:
import pandas as pd

# Clean column names
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Check cluster column exists
if 'cluster' not in df.columns:
    raise ValueError("The dataframe must contain a 'cluster' column.")

# Select numeric columns except cluster
numeric_cols = df.select_dtypes(include='number').columns.tolist()

if 'cluster' in numeric_cols:
    numeric_cols.remove('cluster')

# Group by cluster and compute mean + median
cluster_profile = df.groupby('cluster')[numeric_cols].agg(['mean', 'median'])

# Flatten column names
cluster_profile.columns = ['_'.join(col) for col in cluster_profile.columns]

cluster_profile = cluster_profile.reset_index()

print(cluster_profile)

   cluster  order_id_mean  order_id_median  unit_price_mean  \
0        0    1010.428571           1010.5        25.285714   
1        1    1013.500000           1013.5        26.500000   
2        2    1009.250000           1009.5        23.750000   

   unit_price_median  quantity_mean  quantity_median  total_sales_mean  \
0               25.0       2.357143              2.0         58.642857   
1               26.5       1.500000              1.5         39.000000   
2               23.0       2.250000              2.0         53.500000   

   total_sales_median  
0                55.0  
1                39.0  
2                49.0  


In [23]:
print(df.columns)

Index(['order_id', 'order_date', 'region', 'city', 'product_name',
       'product_category', 'unit_price', 'quantity', 'total_sales', 'cluster'],
      dtype='object')


In [24]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns)

Index(['order_id', 'order_date', 'region', 'city', 'product_name',
       'product_category', 'unit_price', 'quantity', 'total_sales', 'cluster'],
      dtype='object')


In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Load dataset
df = pd.read_csv("data.csv")

# Show dataset structure
print("Columns in dataset:")
print(df.columns)

# Assume the LAST column is the target variable
target_column = df.columns[-1]

# Features and target
X_reg = df.drop(target_column, axis=1)
y_reg = df[target_column]

# Train-test split
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Confirm shapes
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train_reg.shape)
print("Testing target shape:", y_test_reg.shape)

Columns in dataset:
Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')
Training features shape: (455, 32)
Testing features shape: (114, 32)
Training target shape: (455,)
Testing target shape: (114,)


In [33]:
target_column = "Class"

In [34]:
target_column = "is_fraud"

In [35]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [40]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [41]:
print(df.columns)

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')


In [42]:
df.head()
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')